# 02 — Drift Detection Demo

Demonstrates statistical drift detection on real Census ACS data.

**Tests shown:**
- Population Stability Index (PSI)
- Kolmogorov-Smirnov (KS) test
- Alert classification (stable / warning / critical)

In [1]:
import sys
sys.path.insert(0, '..')

from src.model_registry import ModelRegistry
from src.drift_detector import DriftDetector, calculate_psi, ks_test
from src.retrain_pipeline import fetch_census_acs
import pandas as pd
import numpy as np

# Setup
reg = ModelRegistry('../data/registry.db')
detector = DriftDetector(reg)
print("Drift detector ready")

Drift detector ready


## Fetch Real Census Data

In [2]:
# Fetch fresh real data from Census API
df = fetch_census_acs()
print(f"Fetched {len(df)} records")
print(df[['state_name', 'median_income', 'education_rate', 'poverty_rate']].head())

[Census] Fetching ACS data... https://api.census.gov/data/2022/acs/acs5?get=NAME,B01003_001E,B19013_001E,B15003_022E,B15


[Census] Fetched 52 states/territories
Fetched 52 records
   state_name  median_income  education_rate  poverty_rate
0     Alabama          59609        0.166909      0.157225
1      Alaska          86370        0.192940      0.104876
2     Arizona          72581        0.196445      0.130651
3    Arkansas          56335        0.156231      0.162289
4  California          91905        0.221114      0.121243


## Register Production Baseline

Train a simple model and register it as the production baseline.
The baseline distribution is saved for future drift comparisons.

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Simple model: predict median income from demographics
X = df[['population', 'median_age', 'education_rate']].fillna(0)
y = df['median_income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
r2 = model.score(X_test, y_test)

# Register as production baseline
run_id = reg.start_run('census-demographics', 'v1.0', 'production')
reg.log_metric(run_id, 'r2_score', r2)
reg.log_metric(run_id, 'n_samples', len(df))
reg.save_json(run_id, {
    'population': df['population'].describe().to_dict(),
    'median_age': df['median_age'].describe().to_dict(),
    'education_rate': df['education_rate'].describe().to_dict(),
    'poverty_rate': df['poverty_rate'].describe().to_dict(),
}, 'baseline_distribution.json')
reg.save_json(run_id, {'features': ['population', 'median_age', 'education_rate', 'poverty_rate']}, 'metadata.json')
reg.set_tag(run_id, 'stage', 'production')

print(f'Registered production baseline v1.0 (R²={r2:.3f})')
print(f'Run ID: {run_id}')

Registered production baseline v1.0 (R²=0.436)
Run ID: 6


## Run Drift Detection

In [4]:
# Run drift detection against production baseline
features = ['population', 'median_age', 'education_rate', 'poverty_rate']
result = detector.detect_drift_for_experiment('census-demographics', df, features)

print(f"Overall alert: {result['overall_alert'].upper()}")
print("\nFeature-level results:")
for feat, res in result['features'].items():
    print(f"  {feat}:")
    print(f"    PSI: {res['psi']:.4f} ({res['psi_status']})")
    print(f"    KS p-value: {res['ks_p_value']:.4f} ({res['ks_status']})")
    print(f"    Expected μ={res['expected_mean']:.4f}, Actual μ={res['actual_mean']:.4f}")

[Drift] Report saved: /tmp/sierra-genai-engineering/projects/ai-ready-mlops/notebooks/../data/drift/census-demographics_20260520_161716.json
Overall alert: CRITICAL

Feature-level results:
  population:
    PSI: 1.8596 (critical)
    KS p-value: 0.0060 (drift)
    Expected μ=6430191.8269, Actual μ=6430191.8269
  median_age:
    PSI: 0.3823 (critical)
    KS p-value: 0.4508 (stable)
    Expected μ=38.9038, Actual μ=38.9038
  education_rate:
    PSI: 0.1399 (warning)
    KS p-value: 0.9614 (stable)
    Expected μ=0.2061, Actual μ=0.2061
  poverty_rate:
    PSI: 3.4037 (critical)
    KS p-value: 0.0178 (drift)
    Expected μ=0.1293, Actual μ=0.1293


## Manual PSI/KS Test on Single Feature

In [5]:
# Manual test: compare two subsets of the data
subset_a = df[df['median_income'] > df['median_income'].median()]['education_rate'].dropna().values
subset_b = df[df['median_income'] <= df['median_income'].median()]['education_rate'].dropna().values

psi = calculate_psi(subset_a, subset_b)
ks_stat, ks_p = ks_test(subset_a, subset_b)

print(f"Education rate PSI between high/low income states: {psi:.4f}")
print(f"KS test: statistic={ks_stat:.4f}, p-value={ks_p:.4f}")

Education rate PSI between high/low income states: 10.4483
KS test: statistic=0.6154, p-value=0.0001
